In [ ]:
PARAMETERS = globals().get('PARAMETERS', {})
CMD = PARAMETERS['cmd']          # the benchmark command, as a list
CWD = PARAMETERS['cwd']


In [ ]:
# The command's output is read as it comes, and a line is printed at least every five minutes: a cell that prints
# nothing for an hour is a server JupyterHub's idle culler stops (cull.timeout 3600)
import subprocess, sys, threading, time
print(time.strftime('%H:%M:%S'), ' '.join(CMD), flush=True)
p = subprocess.Popen([sys.executable, '-u', *CMD], cwd=CWD, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
lines = []
reader = threading.Thread(target=lambda: lines.extend(p.stdout), daemon=True)
reader.start()
start = time.time()
while p.poll() is None:
    if time.time() - start > 5 * 3600:
        p.kill()
        raise SystemExit('the command ran longer than five hours')
    time.sleep(300)
    if p.poll() is None:
        print(time.strftime('%H:%M:%S'), 'running,', len(lines), 'lines so far', flush=True)
reader.join()
lines = [l.rstrip('\n') for l in lines]
print('\n'.join(l for l in lines if 'RESULT' in l or l.startswith('llebre') or 'adapter' in l or 'Error' in l or 'Traceback' in l or 'rror:' in l)[-8000:])
print(time.strftime('%H:%M:%S'), 'EXIT', p.returncode)
if p.returncode != 0:
    print('\n'.join(lines[-60:]))
    raise SystemExit(p.returncode)
